In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Importing dataset from the competition
import kagglehub

# Download latest version
path = kagglehub.competition_download('arc-prize-2026-arc-agi-2')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/arc-prize-2026-arc-agi-2


## Explaing DSL for ARC

Building a DSL for ARC means I have to create a custom vocabulary of single purpose functions, or they call it "primitives".

Some of the explanation I got online are quite cool. So basically, primitives are like lego bricks. Individually, they serve little purposes. But when a search algo eventually snaps them together (e.g., `rotate(recolor(grid,0,5))`, they can solve complex spatial puzzles.

In the context for ARC, primitives usually fall into 3 categories:
- Colour transformation: Changing specific colours, filling areas, or mapping colour patterns.
- Geometric Transformation: Rotating, flipping, scaling, or cropping grids.
- Object Extraction: Identifying distinct shapes or connected pixels within the grid.

The below is an example of what a simple Colour Transformation and geometric transformation looks like.

To build a DSL vocabulary, I would need to add more bricks to my toolkit.

In [6]:
import numpy as np

# Colour transformations
def primitive_replace_color(grid, old_color, new_color):
    """Replaces all instances of old_color with new_color."""
    new_grid = np.array(grid)
    new_grid[new_grid == old_color] = new_color
    
    return new_grid.tolist()

# Geometric Transformation
def primitive_rotate_90_clockwise(grid):
    """Rotates a 2D grid 90 degrees clockwise."""
    
    # np.rot90 rotates counter-clockwise by default. 
    # Setting k=-1 tells it to rotate clockwise exactly once.
    rotated_grid = np.rot90(grid, k=-1)
    
    return rotated_grid.tolist()

The next one would be creating object transformation, where I will use scipy's amazing image-processing tools: `scipy.ndimage.label`.

In the context of ARC, an "object" is usually an island of coloured pixels touching each other, floating in a sea of black background (zeroes).

The `label` function acts like a flood-fill algo. I give it a grid and it scans through it, finding every distinct island of pixels and assigning each island a unique ID number.

In [5]:
from scipy.ndimage import label

def primitive_find_objects(grid, background_color=0):
    """
    Scans the grid and groups connected pixels into distinct objects, 
    ignoring the background color.
    """
    grid_array = np.array(grid)
    
    # 1. Create a "mask" (True/False grid) that ignores the background
    mask = grid_array != background_color
    
    # 2. structure defines what "connected" means. 
    # This 3x3 cross means pixels must touch up/down/left/right (no diagonals).
    structure = np.array([[0,1,0],
                          [1,1,1],
                          [0,1,0]])
    
    # 3. label() returns the newly labeled grid and the total count of objects
    labeled_array, num_objects = label(mask, structure=structure)
    
    return labeled_array.tolist(), num_objects

With this primitives, instead of looking at a grid of raw colours, your system now knows exactly how many distinct shapes exist in the puzzle. 

If `num_objects` is 3, the search algorithm now knows it has 3 distinct puzzle pieces it can independently move, recolour or delete.

One of the most common mechanics in ARC is moving an object across the grid—think of shapes falling under "gravity" or sliding until they hit a wall.

To build our primitive_move_object function, we would take the isolated object we found using SciPy, calculate those new coordinates for every pixel, and draw them onto a fresh grid.

But when building a robust Domain-Specific Language (DSL), our primitives have to be bulletproof. An AI search algorithm will blindly test hundreds of combinations, and if a primitive crashes, the whole system fails.

If an AI search algorithm tries this move and hits an error, the entire program crashes. To prevent this, every primitive in our DSL needs bounds checking built into it. Before making a move, the code has to verify if the new coordinates actually exist within the grid dimensions.

There are 3 different design choice: 
- **The Wall (Clipping):** The shape moves as far as it can and stops right at the edge, flattening against it.
- **Pac-Man (Wrapping):** The shape disappears off the bottom edge and reappears at the top of the grid.
- **Invalidation (Strict):** The function completely rejects the move and just returns the original, unchanged grid, telling the AI "that's an illegal move.

In the context of an AI search algorithm, **Invalidation** is often the most efficient path. When a search algorithm is hunting for a solution, it generates a massive tree of possible moves. If we allow shapes to wrap around or squish against walls, the AI has to keep evaluating those weird grids to see if they solve the puzzle. By strictly invalidating out-of-bounds moves and returning an error or a None value, we instantly "prune" that branch of the tree, saving massive amounts of compute time.

To build this `primitive_move_object` function, we need to translate that strict invalidation rule into Python logic.

Now I build a function to make a list of the object's coordinates, along with how far we want to move it vertically (`row shift`) and horizontally (`col_shift`).

In [7]:
def primitive_move_object(grid, object_coords, row_shift, col_shift):
    """
    Moves an object (list of (row, col) coordinates) by a specific shift.
    If the move is out of bounds, it returns the original grid unchanged.
    """
    grid_array = np.array(grid)
    height, width = grid_array.shape
    
    new_coords = []
    
    # --- PHASE 1: The Safety Check ---
    for r, c in object_coords:
        new_r = r + row_shift
        new_c = c + col_shift
        
        # The exact boundary logic we built:
        if new_r >= 0 and new_c >= 0 and new_r < height and new_c < width:
            new_coords.append((new_r, new_c))
        else:
            # A pixel fell off the edge! 
            # Reject the entire move and return the original list.
            return grid 
            
    # --- PHASE 2: Execution ---
    # If the code reaches here, the move is 100% valid.
    new_grid_array = np.copy(grid_array)
    
    # 1. Erase the object from its old position
    colors = []
    for r, c in object_coords:
        colors.append(grid_array[r, c]) # Remember the colors
        new_grid_array[r, c] = 0        # Paint it black
        
    # 2. Draw the object in its new position
    for i, (new_r, new_c) in enumerate(new_coords):
        new_grid_array[new_r, new_c] = colors[i]
        
    return new_grid_array.tolist()